# 02 — Feature Engineering

**Goal**: Create, evaluate, and finalize named feature sets used across all modeling phases.  
**Output**: Encoding/scaling strategies defined, 5 named feature sets saved as parquet for reuse.


In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, mutual_info_classif, RFE, SelectFromModel, VarianceThreshold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import json, pathlib

from src.data_utils import (
    load_data, get_X_y, add_engineered_features,
    FEATURE_COLS, TARGET, CONTINUOUS_COLS, BINARY_COLS, ORDINAL_COLS
)
from src.visualization import save_fig, PALETTE

sns.set_theme(style='whitegrid', palette=PALETTE)
RESULTS = pathlib.Path('../results/metrics')
RESULTS.mkdir(exist_ok=True)

train = load_data('train')
X_raw, y = get_X_y(train, extra_features=False)
print(f'Raw X shape: {X_raw.shape}, y shape: {y.shape}')

Raw X shape: (630000, 13), y shape: (630000,)


## 2.1 Encoding Strategy Comparison

In [2]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
probe = LogisticRegression(max_iter=500, random_state=42)

# All features are already encoded as integers — 'label' encoding is the baseline
# Test if StandardScaler helps vs not scaling
print('Quick probe: Effect of scaling on Logistic Regression (3-fold, ROC-AUC)')
print('-' * 55)

scalers = {
    'No scaling': None,
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'QuantileTransformer': QuantileTransformer(output_distribution='normal', random_state=42),
}

scaling_results = {}
for name, scaler in scalers.items():
    if scaler:
        X_fit = scaler.fit_transform(X_raw)
    else:
        X_fit = X_raw.values
    scores = cross_val_score(probe, X_fit, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    scaling_results[name] = scores
    print(f'  {name:<28} ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

Quick probe: Effect of scaling on Logistic Regression (3-fold, ROC-AUC)
-------------------------------------------------------


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solve

  No scaling                   ROC-AUC: 0.9498 ± 0.0004


  StandardScaler               ROC-AUC: 0.9505 ± 0.0001


  RobustScaler                 ROC-AUC: 0.9505 ± 0.0001


  QuantileTransformer          ROC-AUC: 0.9472 ± 0.0001


## 2.2 Engineered Features

In [3]:
train_eng = add_engineered_features(train)
X_eng, y = get_X_y(train_eng, extra_features=True)

print('Engineered feature set shape:', X_eng.shape)
print('New features added:')
new_feats = [c for c in X_eng.columns if c not in FEATURE_COLS]
for f in new_feats:
    print(f'  - {f}')

# MI scores for engineered features
from sklearn.feature_selection import mutual_info_classif
mi_eng = mutual_info_classif(X_eng, y, random_state=42)
mi_eng_df = pd.Series(mi_eng, index=X_eng.columns).sort_values(ascending=False)
print('\nMI rankings for all features (including engineered):')
for feat, score in mi_eng_df.items():
    marker = '← NEW' if feat in new_feats else ''
    print(f'  {feat:<30} {score:.4f}  {marker}')

Engineered feature set shape: (630000, 18)
New features added:
  - Age_HR_ratio
  - BP_Chol_product
  - ST_Slope_interaction
  - High_risk_age
  - ST_abs



MI rankings for all features (including engineered):
  Thallium                       0.2350  
  Chest pain type                0.1890  
  Age_HR_ratio                   0.1402  ← NEW
  Sex                            0.1320  
  Max HR                         0.1286  
  ST_Slope_interaction           0.1246  ← NEW
  Slope of ST                    0.1241  
  Exercise angina                0.1236  
  Number of vessels fluro        0.1208  
  ST_abs                         0.1075  ← NEW
  ST depression                  0.1072  
  EKG results                    0.0764  
  High_risk_age                  0.0691  ← NEW
  Age                            0.0304  
  BP                             0.0117  
  BP_Chol_product                0.0103  ← NEW
  Cholesterol                    0.0101  
  FBS over 120                   0.0016  


## 2.3 Feature Selection Methods

In [4]:
from sklearn.feature_selection import VarianceThreshold

# a) Variance Threshold (remove near-constant features)
scaler_for_vt = StandardScaler()
X_scaled = scaler_for_vt.fit_transform(X_raw)

vt = VarianceThreshold(threshold=0.01)
vt.fit(X_scaled)
kept_vt = X_raw.columns[vt.get_support()].tolist()
dropped_vt = X_raw.columns[~vt.get_support()].tolist()
print(f'Variance Threshold: kept {len(kept_vt)}/{len(X_raw.columns)}')
if dropped_vt:
    print(f'  Dropped: {dropped_vt}')
else:
    print('  All features pass variance threshold')

Variance Threshold: kept 13/13
  All features pass variance threshold


In [5]:
# b) Correlation-based filtering (remove highly correlated pairs)
corr_matrix = X_raw.corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, corr_matrix.loc[row, col])
                   for col in upper.columns
                   for row in upper.index
                   if upper.loc[row, col] > 0.85]

print('Highly correlated feature pairs (|r| > 0.85):')
if high_corr_pairs:
    for f1, f2, r in high_corr_pairs:
        print(f'  {f1} ↔ {f2}  r={r:.3f}')
else:
    print('  None found — all features < 0.85 correlation')

Highly correlated feature pairs (|r| > 0.85):
  None found — all features < 0.85 correlation


In [6]:
# c) SelectKBest (Mutual Information, top 10)
selector_mi = SelectKBest(mutual_info_classif, k=10)
selector_mi.fit(X_raw, y)
mi_selected = X_raw.columns[selector_mi.get_support()].tolist()
print('MI top-10 selected features:', mi_selected)

MI top-10 selected features: ['Age', 'Sex', 'Chest pain type', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


In [7]:
# d) SelectFromModel — Random Forest importance
print('Fitting Random Forest for feature selection (sample=50K)...')
np.random.seed(42)
idx = np.random.choice(len(X_raw), size=50000, replace=False)
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
sfm = SelectFromModel(rf_sel, threshold='mean')
sfm.fit(X_raw.iloc[idx], y.iloc[idx])
rf_selected = X_raw.columns[sfm.get_support()].tolist()

# Feature importances
importances = pd.Series(
    sfm.estimator_.feature_importances_,
    index=X_raw.columns
).sort_values(ascending=False)

print(f'RF SelectFromModel (threshold=mean): {len(rf_selected)} features selected')
print('RF feature importances:')
for feat, imp in importances.items():
    marker = '✓ selected' if feat in rf_selected else ''
    print(f'  {feat:<30} {imp:.4f}  {marker}')

Fitting Random Forest for feature selection (sample=50K)...


RF SelectFromModel (threshold=mean): 6 features selected
RF feature importances:
  Thallium                       0.1754  ✓ selected
  Chest pain type                0.1407  ✓ selected
  Max HR                         0.1348  ✓ selected
  Number of vessels fluro        0.0842  ✓ selected
  Cholesterol                    0.0793  ✓ selected
  ST depression                  0.0784  ✓ selected
  Age                            0.0763  
  Exercise angina                0.0653  
  Slope of ST                    0.0587  
  BP                             0.0575  
  Sex                            0.0287  
  EKG results                    0.0146  
  FBS over 120                   0.0062  


In [8]:
# Visualize RF importances
fig, ax = plt.subplots(figsize=(9, 6))
colors = [sns.color_palette(PALETTE)[1] if f in rf_selected else sns.color_palette(PALETTE)[0]
          for f in importances.sort_values().index]
ax.barh(importances.sort_values().index, importances.sort_values().values, color=colors)
ax.axvline(importances.mean(), color='red', ls='--', lw=1.5, label=f'Mean = {importances.mean():.4f}')
ax.set_title('Random Forest Feature Importances\n(orange = selected by mean threshold)')
ax.set_xlabel('Mean Decrease Impurity')
ax.legend()
fig.tight_layout()
save_fig('02_rf_feature_importances', fig)
plt.show()

## 2.4 PCA Analysis

In [9]:
scaler_pca = StandardScaler()
X_scaled_full = scaler_pca.fit_transform(X_raw)

pca_full = PCA(random_state=42)
pca_full.fit(X_scaled_full)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

n_95 = np.argmax(cumulative >= 0.95) + 1
n_99 = np.argmax(cumulative >= 0.99) + 1
print(f'Components to explain 95% variance: {n_95}')
print(f'Components to explain 99% variance: {n_99}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(explained)+1), explained, color=sns.color_palette(PALETTE)[0])
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumulative)+1), cumulative, 'o-', color=sns.color_palette(PALETTE)[1])
axes[1].axhline(0.95, color='red', ls='--', lw=1, label='95%')
axes[1].axhline(0.99, color='orange', ls='--', lw=1, label='99%')
axes[1].axvline(n_95, color='red', ls=':', lw=1)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance')
axes[1].legend()

fig.tight_layout()
save_fig('02_pca_scree', fig)
plt.show()

Components to explain 95% variance: 12
Components to explain 99% variance: 13


## 2.5 Define & Persist Named Feature Sets

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

FEAT_DIR = pathlib.Path('../results/metrics')

# ─── feat_baseline: raw integer-encoded, no scaling ───────────────────────────
X_baseline = X_raw.copy()

# ─── feat_scaled: StandardScaler ──────────────────────────────────────────────
ss = StandardScaler()
X_scaled_arr = ss.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled_arr, columns=X_raw.columns, index=X_raw.index)

# ─── feat_robust: RobustScaler (outlier-resistant) ────────────────────────────
rs = RobustScaler()
X_robust_arr = rs.fit_transform(X_raw)
X_robust_df = pd.DataFrame(X_robust_arr, columns=X_raw.columns, index=X_raw.index)

# ─── feat_engineered: + interaction terms + StandardScaler ────────────────────
X_eng_full, _ = get_X_y(add_engineered_features(train), extra_features=True)
ss_eng = StandardScaler()
X_eng_arr = ss_eng.fit_transform(X_eng_full)
X_eng_df = pd.DataFrame(X_eng_arr, columns=X_eng_full.columns, index=X_eng_full.index)

# ─── feat_pca: PCA (95% variance) ─────────────────────────────────────────────
pca_95 = PCA(n_components=n_95, random_state=42)
X_pca_arr = pca_95.fit_transform(X_scaled_arr)
X_pca_df = pd.DataFrame(
    X_pca_arr,
    columns=[f'PC{i+1}' for i in range(n_95)],
    index=X_raw.index
)

# ─── feat_selected: RF-selected features, StandardScaler ──────────────────────
X_sel_df = X_scaled_df[rf_selected]

feature_sets = {
    'baseline':   X_baseline,
    'scaled':     X_scaled_df,
    'robust':     X_robust_df,
    'engineered': X_eng_df,
    'pca':        X_pca_df,
    'selected':   X_sel_df,
}

print('Feature sets defined:')
for name, df in feature_sets.items():
    print(f'  {name:<15} shape={df.shape}  cols={list(df.columns)[:5]}...')

Feature sets defined:
  baseline        shape=(630000, 13)  cols=['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol']...
  scaled          shape=(630000, 13)  cols=['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol']...
  robust          shape=(630000, 13)  cols=['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol']...
  engineered      shape=(630000, 18)  cols=['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol']...
  pca             shape=(630000, 12)  cols=['PC1', 'PC2', 'PC3', 'PC4', 'PC5']...
  selected        shape=(630000, 6)  cols=['Chest pain type', 'Cholesterol', 'Max HR', 'ST depression', 'Number of vessels fluro']...


In [11]:
# Quick probe: all feature sets vs Logistic Regression (3-fold)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
probe = LogisticRegression(max_iter=500, random_state=42)

print('Feature Set Probe — LogReg 3-fold ROC-AUC:')
print('-' * 55)
feat_probe_results = {}
for name, Xf in feature_sets.items():
    scores = cross_val_score(probe, Xf, y, cv=cv3, scoring='roc_auc', n_jobs=-1)
    feat_probe_results[name] = {'mean': float(scores.mean()), 'std': float(scores.std())}
    print(f'  {name:<15} ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

Feature Set Probe — LogReg 3-fold ROC-AUC:
-------------------------------------------------------


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solve

/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  baseline        ROC-AUC: 0.9498 ± 0.0004


  scaled          ROC-AUC: 0.9505 ± 0.0001


  robust          ROC-AUC: 0.9505 ± 0.0001


  engineered      ROC-AUC: 0.9506 ± 0.0001


  pca             ROC-AUC: 0.9505 ± 0.0001


  selected        ROC-AUC: 0.9350 ± 0.0000


In [12]:
# Save feature set metadata
feat_meta = {
    'rf_selected_features': rf_selected,
    'mi_top10_features': mi_selected,
    'pca_n_components_95pct': int(n_95),
    'pca_n_components_99pct': int(n_99),
    'high_corr_pairs': [(f1, f2, float(r)) for f1, f2, r in high_corr_pairs],
    'feature_set_dims': {name: df.shape[1] for name, df in feature_sets.items()},
    'feature_set_logreg_auc': feat_probe_results,
    'engineered_features': [c for c in X_eng_full.columns if c not in FEATURE_COLS],
}
pathlib.Path('../results/metrics/02_feature_engineering.json').write_text(
    json.dumps(feat_meta, indent=2)
)
print('Feature engineering metadata saved.')
print(json.dumps(feat_meta, indent=2))

Feature engineering metadata saved.
{
  "rf_selected_features": [
    "Chest pain type",
    "Cholesterol",
    "Max HR",
    "ST depression",
    "Number of vessels fluro",
    "Thallium"
  ],
  "mi_top10_features": [
    "Age",
    "Sex",
    "Chest pain type",
    "EKG results",
    "Max HR",
    "Exercise angina",
    "ST depression",
    "Slope of ST",
    "Number of vessels fluro",
    "Thallium"
  ],
  "pca_n_components_95pct": 12,
  "pca_n_components_99pct": 13,
  "high_corr_pairs": [],
  "feature_set_dims": {
    "baseline": 13,
    "scaled": 13,
    "robust": 13,
    "engineered": 18,
    "pca": 12,
    "selected": 6
  },
  "feature_set_logreg_auc": {
    "baseline": {
      "mean": 0.9497840165453614,
      "std": 0.00037061346417287494
    },
    "scaled": {
      "mean": 0.950492052699949,
      "std": 0.00010355653391854247
    },
    "robust": {
      "mean": 0.9504921872568332,
      "std": 0.00010407420586104713
    },
    "engineered": {
      "mean": 0.95058203451827